In [0]:
# ==========================================
# Project: Retail Medallion Architecture
# Layer: Gold
# Purpose: Top Products KPI
# ==========================================

from pyspark.sql import functions as F

# ------------------------------------------
# Configuration
# ------------------------------------------

SILVER_TABLE = "retail_project.silver.online_retail"
GOLD_TABLE = "retail_project.gold.top_products"

# ------------------------------------------
# Read Silver Data
# ------------------------------------------

df_silver = spark.table(SILVER_TABLE)

# ------------------------------------------
# Revenue Calculation
# ------------------------------------------

df_gold_base = (
    df_silver
    .withColumn(
        "revenue",
        F.round(
            F.col("Quantity") * F.col("UnitPrice"),
            2
        )
    )
)

# ------------------------------------------
# Product Aggregation
# ------------------------------------------

top_products = (
    df_gold_base
    .groupBy(
        "StockCode",
        "Description"
    )
    .agg(
        F.round(
            F.sum("revenue"),
            2
        ).alias("total_revenue")
    )
    .filter(
        ~F.col("StockCode").isin("DOT", "POST")
    )
    .orderBy(
        F.desc("total_revenue")
    )
)

# ------------------------------------------
# Validation
# ------------------------------------------

top_products.show(20, False)

# ------------------------------------------
# Write Gold Table
# ------------------------------------------

(
    top_products.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_TABLE)
)

print("Top Products Gold table created successfully.")